# Prepare Library & Environment

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# OpenAI API key
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "") 

# MongoDB connection string
MONGO_URI = os.getenv("MONGO_URI", "")

# MongoDB database to store the knowledge graph
MONGO_DB = os.getenv("MONGO_DB", "")  

# MongoDB collection to store the knowledge graph
MONGO_GRAPH_COLLECTION = os.getenv("MONGO_GRAPH_COLLECTION", "")

# MongoDB collection to store the vector embeddings
MONGO_VECTOR_COLLECTION = os.getenv("MONGO_VECTOR_COLLECTION", "") 

# MongoDB collection to store the vector index
MONGO_VECTOR_INDEX = os.getenv("MONGO_VECTOR_INDEX", "")  

# Document Preparation

## Download PDF

In [2]:
import requests

def download_pdf(arxiv_pdf_url: str, output_path: str):
    """
    Download a PDF from the given arXiv PDF URL and save to output_path.
    """
    response = requests.get(arxiv_pdf_url, stream=True)
    response.raise_for_status()  # will raise an exception for HTTP errors

    with open(output_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:  # filter out keep-alive chunks
                f.write(chunk)

In [4]:
!mkdir papers
download_pdf(arxiv_pdf_url="https://arxiv.org/pdf/2506.09985", output_path="papers/V-JEPA2.pdf")
download_pdf(arxiv_pdf_url="https://arxiv.org/pdf/2509.02722", output_path="papers/VLWM.pdf")

mkdir: papers: File exists


## Text Splitting

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document

def load_and_split_pdf(pdf_path: str):
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()

    # Split pages into overlapping chunks for better entity extraction context
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1200, chunk_overlap=200, separators=["\n\n", "\n", " ", ""]
    )
    splits = splitter.split_documents(pages)
    # Attach lightweight provenance metadata
    enriched = []
    for i, d in enumerate(splits):
        md = dict(d.metadata or {})
        md.update(
            {
                "source": os.path.basename(pdf_path),
                "chunk_id": i,
                "page": md.get("page"),
            }
        )
        enriched.append(Document(page_content=d.page_content, metadata=md))
    return enriched

In [5]:
pdf_file_paths = [
    'papers/V-JEPA2.pdf',
    'papers/VLWM.pdf'
]

paper_docs = []
for pdf_path in pdf_file_paths:
    paper_docs += load_and_split_pdf(pdf_path)
paper_docs

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:)', 'creationdate': '', 'author': 'Mido Assran; Adrien Bardes; David Fan; Quentin Garrido; Russell Howes; Mojtaba; Komeili; Matthew Muckley; Ammar Rizvi; Claire Roberts; Koustuv Sinha; Artem Zholus; Sergio Arnaud; Abha Gejji; Ada Martin; Francois Robert Hogan; Daniel Dugas; Piotr Bojanowski; Vasil Khalidov; Patrick Labatut; Francisco Massa; Marc Szafraniec; Kapil Krishnakumar; Yong Li; Xiaodong Ma; Sarath Chandar; Franziska Meier; Yann LeCun; Michael Rabbat; Nicolas Ballas', 'doi': 'https://doi.org/10.48550/arXiv.2506.09985', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'title': 'V-JEPA 2: Self-Supervised Video Models Enable Understanding, Prediction and Planning', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2506.09985v1', 'source': 'V-JEPA2.pdf', 'total_pages': 48, 'page'

## Text Chunk to VectorDB

In [8]:
from langchain_mongodb.vectorstores import MongoDBAtlasVectorSearch
from langchain_openai import OpenAIEmbeddings
from pymongo import MongoClient

embeddings = OpenAIEmbeddings(model="text-embedding-3-large", api_key=OPENAI_API_KEY)
client = MongoClient(MONGO_URI)

graph_collection = client[MONGO_DB][MONGO_VECTOR_COLLECTION]

# Create the vector store wrapper
vector_store = MongoDBAtlasVectorSearch(
        collection=graph_collection,
        embedding=embeddings,
        index_name=MONGO_VECTOR_INDEX,       # must match an existing Atlas Vector Search index
        text_key="text",             # field to store raw text
        embedding_key="embedding",   # field to store vector
    )
    # Upsert chunk texts + metadata
chuncks_id = vector_store.add_documents(paper_docs)
len(chuncks_id)

291

## Create Index Fields for VectorSearch

In [ ]:
from pymongo.operations import SearchIndexModel

definition={
    "fields": [
        {
            "type": "vector",
            "numDimensions": 3072, # OpenAI text-embedding-3-Large = 3072
            "path": "embedding",
            "similarity": "dotProduct"
        }
    ]
}

search_index_model = SearchIndexModel(
    definition=definition,
    name=MONGO_VECTOR_INDEX,
    type="vectorSearch"
)

vector_collection = MongoClient(MONGO_URI)[MONGO_DB][MONGO_VECTOR_COLLECTION]
try:
    vector_collection.create_search_index(model=search_index_model)
except Exception as e:
    vector_collection.update_search_index(MONGO_VECTOR_INDEX, definition)

## Prepare GraphStore

In [9]:
from langchain_openai import OpenAI
from langchain.chat_models import init_chat_model

chat_model = init_chat_model("gpt-4o", model_provider="openai", api_key=OPENAI_API_KEY, temperature=0)

In [10]:
from langchain_mongodb.graphrag.graph import MongoDBGraphStore

graph_store = MongoDBGraphStore(
    connection_string=MONGO_URI,
    database_name=MONGO_DB,
    collection_name=MONGO_GRAPH_COLLECTION,
    entity_extraction_model = chat_model,
    allowed_relationship_types=["cites", "extends", "uses", "related_to", "authored_by"],
)

In [11]:
from tqdm import tqdm
import re

def normalize_entity(name: str) -> str:
    """Normalize entity name for consistency in GraphRAG."""
    name = name.strip()

    # 1️⃣ Normalize person names: "Alice Nguyen" → "Alice N."
    parts = name.split()
    if len(parts) == 2 and parts[0][0].isupper() and parts[1][0].isupper():
        first, last = parts
        return f"{first} {last[0]}."

    # 2️⃣ Normalize model or object names: "cosmos model" → "cosmos"
    name = re.sub(r"\b(model|system|framework|network|dataset)\b", "", name, flags=re.IGNORECASE).strip()

    # 3️⃣ Lowercase normalization for technical terms (optional)
    if any(ch.islower() for ch in name):
        name = name.lower()

    return name

def add_documents(graph_store, documents, vectors=[]):
    documents = [documents] if not isinstance(documents, list) else documents
    results = []
    vector_ids = None
    if len(documents) == len(vectors):
        vector_ids = vectors.copy()
    for i_, doc in enumerate(tqdm(documents)):
        try:
            entities = graph_store.extract_entities(doc.page_content)
            for e in entities:
                # e["_id"] = normalize_entity(e["_id"])
                if vector_ids:
                    if "attributes" in e:
                        e["attributes"]["vector_id"] = [vector_ids[i_]]
                    else:
                        e["attributes"] = {"vector_id": [vector_ids[i_]]}
        except Exception as e:
            # print(f"[skip1: parse error] {e}")
            continue
        if not entities:
            # print("[skip: no entities]")
            continue
        try:
            res = graph_store._write_entities(entities)
            if res is not None:
                results.append(res)
        except Exception as e:
            # print(f"[skip2: parse error] {e}")
            continue
    return results

In [ ]:
add_documents(graph_store, paper_docs, chuncks_id)

In [19]:
import networkx as nx
import pandas as pd
from pyvis.network import Network

filter_out_attributes = ["authored_by", "authors", "vector_id"]
filter_out_types = ["Person"]

def visualize_graph(collection, output_path):
    docs = list(collection.find())
    palette = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
        "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
        "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173"
    ]
    
    df = pd.DataFrame(docs)
    all_type = df["type"].unique()
    color_map = {c: palette[i % len(palette)] for i, c in enumerate(all_type)}

    def format_attributes(attrs):
        return "<br>".join(f"{k}: {', '.join(v)}" for k, v in attrs.items()) if attrs else ""
    
    G = nx.DiGraph()

    # Create nodes
    for doc in docs:
        node_id = str(doc["_id"])
        info = f"Type: {doc.get('type', '')}"
        if doc.get('type', '') not in filter_out_types and "type" in doc:
            if "attributes" in doc:
                attr_info = format_attributes(doc["attributes"])
                if attr_info:
                    info += "<br>" + attr_info
            
            G.add_node(node_id, label=node_id, title=info.replace("<br>", "\n"), color=color_map.get(doc["type"]),)

    # Create edges
    for doc in docs:
        source = str(doc["_id"])
        rels = doc.get("relationships", {})
        targets = rels.get("target_ids", [])
        types = rels.get("types", [])
        attrs = rels.get("attributes", [])
        
        for i, target in enumerate(targets):
            edge_type = types[i] if i < len(types) else ""
            extra = attrs[i] if i < len(attrs) else {}
            edge_info = f"Relationship: {edge_type}"
            if extra:
                edge_info += "<br>" + format_attributes(extra)
            if edge_type not in filter_out_attributes:
                G.add_edge(source, str(target), label=edge_type, title=edge_info.replace("<br>", "\n"))

        node_attrs = doc.get("attributes", {})
        for sub_att, v_sub in node_attrs.items():
            if sub_att not in filter_out_attributes:
                for v in v_sub:
                    G.add_edge(source, str(v), label=sub_att)

    # Build and configure network
    nt = Network(height="750px", width="100%", directed=True, notebook=True)
    nt.from_nx(G)
    nt.show(output_path)

In [21]:
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
graph_collection = client[MONGO_DB][MONGO_GRAPH_COLLECTION]
visualize_graph(graph_collection, "knowledgeGraph.html")

knowledgeGraph.html


In [22]:
!open "knowledgeGraph.html"